## 1. Analise de metadata/schema arquivos parquet

### a) Imports e path

In [0]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, TimestampType, StringType

# -- Path -- #
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(project_root)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Workspace/Repos/pierinicoelho@gmail.com


### b) Primeiras linhas (Janeiro-2023)

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import RAW_VOLUME_PATH

january_path = f"{RAW_VOLUME_PATH}/yellow/yellow_tripdata_2023-01.parquet"

print(f"Lendo dados raw do mês 01...\nCaminho: {january_path}")

df_raw_january = spark.read.parquet(january_path)
col_qty = len(df_raw_january.columns)

print(f"Total de registros em Janeiro, Qtd de Linhas: {df_raw_january.count():,} | Qtd de colunas: {col_qty}\n")
print("Exibindo as 10 primeiras linhas:")
display(df_raw_january.limit(10))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Lendo dados raw do mês 01...
Caminho: /Volumes/workspace/ifood_taxi_case/landingzone_raw_taxis/yellow/yellow_tripdata_2023-01.parquet
Total de registros em Janeiro, Qtd de Linhas: 3,066,766 | Qtd de colunas: 19

Exibindo as 10 primeiras linhas:


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
2,2023-01-01T00:32:10.000,2023-01-01T00:40:36.000,1.0,0.97,1.0,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
2,2023-01-01T00:55:08.000,2023-01-01T01:01:27.000,1.0,1.1,1.0,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2023-01-01T00:25:04.000,2023-01-01T00:37:49.000,1.0,2.51,1.0,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0
1,2023-01-01T00:03:48.000,2023-01-01T00:13:25.000,0.0,1.9,1.0,N,138,7,1,12.1,7.25,0.5,0.0,0.0,1.0,20.85,0.0,1.25
2,2023-01-01T00:10:29.000,2023-01-01T00:21:19.000,1.0,1.43,1.0,N,107,79,1,11.4,1.0,0.5,3.28,0.0,1.0,19.68,2.5,0.0
2,2023-01-01T00:50:34.000,2023-01-01T01:02:52.000,1.0,1.84,1.0,N,161,137,1,12.8,1.0,0.5,10.0,0.0,1.0,27.8,2.5,0.0
2,2023-01-01T00:09:22.000,2023-01-01T00:19:49.000,1.0,1.66,1.0,N,239,143,1,12.1,1.0,0.5,3.42,0.0,1.0,20.52,2.5,0.0
2,2023-01-01T00:27:12.000,2023-01-01T00:49:56.000,1.0,11.7,1.0,N,142,200,1,45.7,1.0,0.5,10.74,3.0,1.0,64.44,2.5,0.0
2,2023-01-01T00:21:44.000,2023-01-01T00:36:40.000,1.0,2.95,1.0,N,164,236,1,17.7,1.0,0.5,5.68,0.0,1.0,28.38,2.5,0.0
2,2023-01-01T00:39:42.000,2023-01-01T00:50:36.000,1.0,3.01,1.0,N,141,107,2,14.9,1.0,0.5,0.0,0.0,1.0,19.9,2.5,0.0


### c) Analise datatypes por mes
Objetivo: Mapear schema drift entre as partições mensais. \
O auto-detect do Spark é insuficiente para o upcasting necessário, sendo indispensável a padronização via Schema Enforcement para evitar que colunas com tipos conflitantes (ex: Double vs Long) causem falhas na unificação.

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import RAW_VOLUME_PATH, TARGET_YEARS, TARGET_MONTHS

print("Extraindo metadados dos arquivos dinamicamente...\n")

schema_dict = {}

for year in TARGET_YEARS:
    for month in TARGET_MONTHS:
        formatted_month = str(month).zfill(2)
        database_path = f"{RAW_VOLUME_PATH}/yellow/yellow_tripdata_{year}-{formatted_month}.parquet"
        period_key = f"{year}_{formatted_month}"
        
        try:
            file_schema = spark.read.parquet(database_path).schema
            schema_dict[period_key] = {
                data_field.name: data_field.dataType.simpleString() 
                for data_field in file_schema.fields
            }
        except Exception as e:
            print(f"Erro ao ler o arquivo do período {period_key}. Verifique se o caminho esta correto. Erro: {e}")

df_comparison = pd.DataFrame(schema_dict)
df_comparison = df_comparison.fillna("--- AUSENTE ---")

df_comparison = df_comparison.reset_index()
df_comparison = df_comparison.rename(columns={'index': 'column_name'})

period_columns = [col for col in df_comparison.columns if col != 'column_name']

type_conflicts = df_comparison[df_comparison[period_columns].nunique(axis=1) > 1]

if not type_conflicts.empty:
    print("ATENÇÃO! Foram encontrados conflitos de tipos nas seguintes colunas:")
    display(type_conflicts)
else:
    print("Todos os arquivos têm exatamente o mesmo esquema e tipos de dados!")

print("\n--- Esquema completo de todos os períodos ---")
display(df_comparison)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Extraindo metadados dos arquivos dinamicamente...

ATENÇÃO! Foram encontrados conflitos de tipos nas seguintes colunas:


column_name,2023_01,2023_02,2023_03,2023_04,2023_05
VendorID,bigint,int,int,int,int
passenger_count,double,bigint,bigint,bigint,bigint
RatecodeID,double,bigint,bigint,bigint,bigint
PULocationID,bigint,int,int,int,int
DOLocationID,bigint,int,int,int,int
airport_fee,double,--- AUSENTE ---,--- AUSENTE ---,--- AUSENTE ---,--- AUSENTE ---
Airport_fee,--- AUSENTE ---,double,double,double,double



--- Esquema completo de todos os períodos ---


column_name,2023_01,2023_02,2023_03,2023_04,2023_05
VendorID,bigint,int,int,int,int
tpep_pickup_datetime,timestamp_ntz,timestamp_ntz,timestamp_ntz,timestamp_ntz,timestamp_ntz
tpep_dropoff_datetime,timestamp_ntz,timestamp_ntz,timestamp_ntz,timestamp_ntz,timestamp_ntz
passenger_count,double,bigint,bigint,bigint,bigint
trip_distance,double,double,double,double,double
RatecodeID,double,bigint,bigint,bigint,bigint
store_and_fwd_flag,string,string,string,string,string
PULocationID,bigint,int,int,int,int
DOLocationID,bigint,int,int,int,int
payment_type,bigint,bigint,bigint,bigint,bigint


### d) Teste Schema Enforcement
Conclusão: Para as 5 colunas mapeadas com diferenças, testamos upcasting com schema enforcement. \
O teste foi bem-sucedido.

In [0]:
%load_ext autoreload
%autoreload 2

from src.config.settings import RAW_VOLUME_PATH

bronze_schema = StructType([
    StructField("VendorID", LongType(), True), #<-- Diferença de schema
    StructField("tpep_pickup_datetime", TimestampType(), True), 
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True), #<-- Diferença de schema
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True), #<-- Diferença de schema
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True), #<-- Diferença de schema
    StructField("DOLocationID", LongType(), True), #<-- Diferença de schema
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])

print("Lendo a base Bronze com Schema Enforcement...")
df_silver = (
    spark.read
    .schema(bronze_schema)
    .parquet(f"{RAW_VOLUME_PATH}/yellow/*.parquet")
)


rows_qty_silver = df_silver.count()
col_qty_silver = len(df_silver.columns)
print(f"Total de registros em Janeiro, Qtd de Linhas: {df_silver.count():,} | Qtd de colunas: {col_qty}\n")
print("\n10 primeiras linhas do DataFrame unificado:")
display(df_silver.limit(10))
conflict_columns = ["VendorID", "passenger_count", "RatecodeID", "PULocationID", "DOLocationID"]
print("\nColunas que possuíam conflitos de tipo:")
display(df_silver.select(*conflict_columns).limit(10))



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Lendo a base Bronze com Schema Enforcement...
Total de registros em Janeiro, Qtd de Linhas: 16,186,386 | Qtd de colunas: 19


10 primeiras linhas do DataFrame unificado:


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
1,2023-05-01T00:33:13.000Z,2023-05-01T00:53:01.000Z,0,7.8,1,N,138,43,1,33.8,7.75,0.5,8.6,0.0,1.0,51.65,0.0,1.75
1,2023-05-01T00:42:49.000Z,2023-05-01T01:11:18.000Z,2,8.1,1,N,138,262,1,35.9,10.25,0.5,9.5,0.0,1.0,57.15,2.5,1.75
1,2023-05-01T00:56:34.000Z,2023-05-01T01:13:39.000Z,2,9.1,1,N,138,141,1,35.2,10.25,0.5,10.7,6.55,1.0,64.2,2.5,1.75
2,2023-05-01T00:00:52.000Z,2023-05-01T00:20:12.000Z,1,8.21,1,N,138,140,1,33.1,6.0,0.5,2.24,0.0,1.0,47.09,2.5,1.75
1,2023-05-01T00:05:50.000Z,2023-05-01T00:19:41.000Z,0,7.9,1,N,138,263,1,31.0,10.25,0.5,9.85,6.55,1.0,59.15,2.5,1.75
1,2023-05-01T00:42:54.000Z,2023-05-01T01:04:49.000Z,0,10.4,1,N,138,246,1,42.2,10.25,0.5,8.5,6.55,1.0,69.0,2.5,1.75
2,2023-05-01T00:50:34.000Z,2023-05-01T01:12:09.000Z,1,9.05,1,N,138,116,1,38.0,6.0,0.5,10.76,6.55,1.0,64.56,0.0,1.75
1,2023-05-01T00:13:58.000Z,2023-05-01T00:18:10.000Z,1,0.7,1,N,161,48,1,6.5,3.5,0.5,2.85,0.0,1.0,14.35,2.5,0.0
2,2023-04-30T23:48:31.000Z,2023-04-30T23:57:35.000Z,1,2.38,1,N,249,231,2,12.1,1.0,0.5,0.0,0.0,1.0,17.1,2.5,0.0
2,2023-05-01T00:28:47.000Z,2023-05-01T00:39:33.000Z,1,2.92,1,N,114,230,2,14.9,1.0,0.5,0.0,0.0,1.0,19.9,2.5,0.0



Colunas que possuíam conflitos de tipo:


VendorID,passenger_count,RatecodeID,PULocationID,DOLocationID
1,0,1,138,43
1,2,1,138,262
1,2,1,138,141
2,1,1,138,140
1,0,1,138,263
1,0,1,138,246
2,1,1,138,116
1,1,1,161,48
2,1,1,249,231
2,1,1,114,230
